In [3]:
!pip install transformers datasets torch scikit-learn

import torch
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments

dataset = load_dataset("emotion")

train_dataset = dataset["train"].shuffle(seed=24).select(range(150))
test_dataset = dataset["test"].shuffle(seed=24).select(range(40))

labels = ["sadness","joy","love","anger","fear","surprise"]
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)
train_dataset.set_format(type="torch", columns=["input_ids","attention_mask","label"])
test_dataset.set_format(type="torch", columns=["input_ids","attention_mask","label"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=6
)
model.to(device)

training_args = TrainingArguments(
    output_dir="./results",
    max_steps=20,
    per_device_train_batch_size=8,
    logging_steps=1
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

def predict_emotion(text):

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()} # Move inputs to the same device as the model

    model.eval() # Set model to evaluation mode
    with torch.no_grad(): # Disable gradient calculation for inference
        outputs = model(**inputs)

    probs = torch.nn.functional.softmax(outputs.logits, dim=1)

    prediction = torch.argmax(probs).item()

    return labels[prediction]

print("\nPredictions:\n")

print("Text: I am feeling extremely joyful today!")
print("Emotion:", predict_emotion("I am feeling extremely joyful today!"))

print("\nText: I am scared of the dark")
print("Emotion:", predict_emotion("I am scared of the dark"))

print("\nText: I just found love in my life")
print("Emotion:", predict_emotion("I just found love in my life"))

print("\nText: I am furious about the situation")
print("Emotion:", predict_emotion("I am furious about the situation"))

print("\nText: That surprise party shocked me")
print("Emotion:", predict_emotion("That surprise party shocked me"))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
1,1.671781
2,1.798602
3,1.839771
4,1.815627
5,2.073535
6,1.663469
7,1.836473
8,1.923819
9,1.611296
10,1.544698


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Predictions:

Text: I am feeling extremely joyful today!
Emotion: sadness

Text: I am scared of the dark
Emotion: sadness

Text: I just found love in my life
Emotion: sadness

Text: I am furious about the situation
Emotion: sadness

Text: That surprise party shocked me
Emotion: sadness
